# 2. Large results

A TAP result that exceeds the inline caps (200 rows or 48 KiB by default) is
not returned inline and is never stored by the server. Instead `run_adql_query`
in `mode="auto"` re-submits the query as an async job at the archive and hands
back the job's URL. The client polls, then fetches the result bytes itself
with pyvo. This notebook walks that whole path.

Requirements: `pip install manna-mcp pandas` and network access. pandas is
needed only by the save recipe at the end.

In [1]:
import asyncio

from fastmcp import Client

from manna.app import build_mcp

client = Client(build_mcp())
# client = Client("http://localhost:8000/mcp/")  # a server started with `python -m manna`


def payload(result):
    """The tool's JSON envelope. fastmcp exposes it as structured_content."""
    return result.structured_content


ALMA_TAP = "https://almascience.nrao.edu/tap"

## A query that does not fit inline

`TOP 1000` is five times the default inline row cap. With `mode="auto"` the
tool runs the query once synchronously, finds the result exceeds the cap, and
re-submits it as an async job, returning a promotion envelope instead of
rows. The first execution is the cost of not knowing the result size up
front.

In [2]:
adql = "SELECT TOP 1000 obs_id, target_name, s_ra, s_dec, band_list FROM ivoa.obscore"

async with client:
    promoted = payload(
        await client.call_tool(
            "run_adql_query", {"endpoint": ALMA_TAP, "adql": adql, "mode": "auto"}
        )
    )

print("mode:", promoted["mode"], "| phase:", promoted["phase"])
print("job_url:", promoted["job_url"])
print()
print("next_steps:", promoted["next_steps"])

mode: async | phase: EXECUTING
job_url: https://almascience.nrao.edu/tap/async/qd8q5809b9z76ivl

next_steps: ['Poll get_async_job_status(job_url) until phase is COMPLETED or ERROR — pass back the job_url from this response, verbatim.', 'When COMPLETED, call get_async_job_results(job_url) to get the result_url and a fetch_recipe.', 'Then execute the fetch_recipe code with your code-execution tool to load the data — do not abandon the job or re-submit the query.']


## Poll the job

Jobs are addressed by their upstream `job_url`; the server keeps no registry.
`get_async_job_status` asks the archive directly.

In [3]:
job_url = promoted["job_url"]

async with client:
    while True:
        status = payload(await client.call_tool("get_async_job_status", {"job_url": job_url}))
        print(status["phase"])
        if status["phase"] in ("COMPLETED", "ERROR", "ABORTED"):
            break
        await asyncio.sleep(3)

status

EXECUTING


COMPLETED


{'job_url': 'https://almascience.nrao.edu/tap/async/qd8q5809b9z76ivl',
 'phase': 'COMPLETED',
 'started_at': None,
 'ended_at': None,
 'error_message': None,
 'archive': 'alma'}

## Fetch the result yourself

`get_async_job_results` does not return rows either. It returns the archive's
`result_url` plus a `fetch_recipe`: a pyvo snippet that loads the result in
your own environment as an `astropy.table.Table` named `table`.

In [4]:
async with client:
    results = payload(await client.call_tool("get_async_job_results", {"job_url": job_url}))

print("result_url:", results["result_url"])
print()
print(results["fetch_recipe"]["code"])

result_url: https://almascience.nrao.edu/tap/files/result_qd8q5809b9z76ivl.xml

import pyvo
job = pyvo.dal.AsyncTAPJob('https://almascience.nrao.edu/tap/async/qd8q5809b9z76ivl')
job.raise_if_error()
table = job.fetch_result().to_table()


In [5]:
ns = {}
exec(results["fetch_recipe"]["code"], ns)  # the snippet defines `table`
table = ns["table"]

print(len(table), "rows,", len(table.colnames), "columns")
table[:5]

1000 rows, 5 columns


obs_id,target_name,s_ra,s_dec,band_list
,,deg,deg,
str64,str256,float64,float64,str30
uid://A001/X3645/X90.source.S3_70.spw.25,S3_70,149.9615125000217,2.0369822222224983,6
uid://A001/X3645/X90.source.S3_70.spw.29,S3_70,149.9615125000217,2.0369822222224983,6
uid://A001/X3645/X90.source.S3_70.spw.27,S3_70,149.9615125000217,2.0369822222224983,6
uid://A001/X3645/X90.source.S3_70.spw.31,S3_70,149.9615125000217,2.0369822222224983,6
uid://A001/X2d20/X231a.source.J1634-2058.spw.27,J1634-2058,248.6263489170075,-20.973871780600177,6


## Save it so you never run it twice

The envelope also carries `save_recipe`: a snippet that writes the rows to
`manna_cache/<fingerprint>.csv` and appends one line to
`manna_cache/catalog.csv`. It expects a pandas DataFrame named `df`. Whether
a client consults the catalog before re-running a query is the client's
policy; the server only provides the recipe.

In [6]:
from pathlib import Path

df = table.to_pandas()
exec(results["save_recipe"]["code"], {"df": df})

for p in sorted(Path("manna_cache").iterdir()):
    print(p, p.stat().st_size, "bytes")
print()
print(Path("manna_cache/catalog.csv").read_text())

manna_cache/0aea4a18bd42.csv 98446 bytes
manna_cache/catalog.csv 343 bytes

"fingerprint","tool","endpoint","archive","query","target","n_rows","truncated","maxrec","csv_path","saved_at"
"0aea4a18bd42","tap","https://almascience.nrao.edu/tap","alma","SELECT TOP 1000 obs_id, target_name, s_ra, s_dec, band_list FROM ivoa.obscore","","1000","False","","manna_cache/0aea4a18bd42.csv","2026-09-16T20:54:00.381111+00:00"



Clean up the cache directory this notebook created (a real client would
keep it).

In [7]:
import shutil

shutil.rmtree("manna_cache")

Next: {doc}`03-images-and-catalogs` uses the workflow tools that bundle
resolve-then-search into one call.